# Lab 04: HR Employee Dataset — data cleaning with pandas

In [1]:
import pandas as pd
import numpy as np

# Raw HR employee dataset created for data cleaning practice.
employees = pd.DataFrame({
    "EmployeeID": [101, 102, 103, 103, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115],
    "FirstName": ["Aizura", "Bakyt", "Gulnara", "Gulnara", "Nurlan", "Aigul", "Beksultan", "Aisuluu",
                  "Ermek", "Nurgul", "Daniyar", "Aizhan", "Rubat", "Parizat", "Adilet"],
    "LastName": ["Asanova", "Kyzyloy", "Mamatova", "Mamatova", "Abdyldaev", "Sultanova", "Toktogulov",
                 "Baisalova", "Kadyrov", "Jumabaeva ", "Turgunov", "Ryskulova", "Kadyraliev",
                 "Bektybekova", "KOLDOKULOV"],
    "Department": ["it", "HR", "it", "IT", "Finance", "HR", " IT ", "Finance", "IT", "HR", "It",
                   "Financing", "Information Tecnology", "HR", "finance"],
    "Gender": ["F", "m", "F", "f", "M", "F", "M", "F", "M", "F", "M", "F", "F", "M", "m"],
    "Age": ["28", "35", "29", "29", "150", "31", "33", "27", "30", "38", "26", "34", "29", "32", "31"],
    "Salary": ["80000", "65000", np.nan, np.nan, "90000", "70000", "unknown", "62000", "85000", "58000",
               "72000", "66000", "78000", "88000", "61000"],
    "HireDate": ["2023-01-15", "2022-06-10", "2024-03-01", "2024-03-01", "2021-11-20", "2023-08-05",
                 "2020-05-12", "2022-09-18", "2021-03-25", "2019-07-14", "2023-04-18", "2020-10-02",
                 "2022-12-11", "2021-08-30", "2023-06-22"],
    "PerformanceScore": [4.2, 3.8, 4.5, 4.5, 4.0, 3.9, 4.1, 3.7, 4.3, 3.6, 4.4, 3.8, 4.2, "4.0", 3.9],
})

departments = pd.DataFrame({
    "DepartmentID": [1, 2, 3],
    "DepartmentName": ["IT", "HR", "Finance"],
    "Location": ["Bishkek", "Osh", "Bishkek"],
})

## Task 1 — Inspect the dataset

In [2]:
print(employees.shape)
employees.info()
print(employees[["Salary", "Age", "HireDate"]].dtypes)

(15, 9)
<class 'pandas.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   EmployeeID        15 non-null     int64 
 1   FirstName         15 non-null     str   
 2   LastName          15 non-null     str   
 3   Department        15 non-null     str   
 4   Gender            15 non-null     str   
 5   Age               15 non-null     str   
 6   Salary            13 non-null     str   
 7   HireDate          15 non-null     str   
 8   PerformanceScore  15 non-null     object
dtypes: int64(1), object(1), str(7)
memory usage: 1.2+ KB
Salary      str
Age         str
HireDate    str
dtype: object


In [3]:
# Rows where Salary is present (notna)
print(employees[employees["Salary"].notna()].shape[0], "rows have a Salary value")

# Data-quality problems
print("Missing values:\n", employees.isna().sum())
print("Duplicated EmployeeID:\n", employees[employees["EmployeeID"].duplicated(keep=False)][["EmployeeID", "FirstName"]])
print("Department variants:", employees["Department"].unique())
print("Non-numeric Salary:", employees.loc[pd.to_numeric(employees["Salary"], errors="coerce").isna() & employees["Salary"].notna(), "Salary"].tolist())
print("Impossible Age (>100):", employees.loc[pd.to_numeric(employees["Age"]) > 100, "Age"].tolist())
print("HireDate dtype:", employees["HireDate"].dtype, "(text, not datetime)")

13 rows have a Salary value
Missing values:
 EmployeeID          0
FirstName           0
LastName            0
Department          0
Gender              0
Age                 0
Salary              2
HireDate            0
PerformanceScore    0
dtype: int64
Duplicated EmployeeID:
    EmployeeID FirstName
2         103   Gulnara
3         103   Gulnara
Department variants: <StringArray>
[                   'it',                    'HR',                    'IT',
               'Finance',                  ' IT ',                    'It',
             'Financing', 'Information Tecnology',               'finance']
Length: 9, dtype: str
Non-numeric Salary: ['unknown']
Impossible Age (>100): ['150']
HireDate dtype: str (text, not datetime)


In [4]:
# Remove extra spaces from every text column (only real strings; numbers stay as they are)
for col in employees.select_dtypes(include="object"):
    employees[col] = employees[col].apply(lambda v: v.strip() if isinstance(v, str) else v)

# PerformanceScore is mixed: floats plus the string "4.0"
print(employees["PerformanceScore"].map(type).value_counts())
print("All numeric?", pd.to_numeric(employees["PerformanceScore"], errors="coerce").notna().all())
employees["PerformanceScore"] = pd.to_numeric(employees["PerformanceScore"])

PerformanceScore
<class 'float'>    14
<class 'str'>       1
Name: count, dtype: int64
All numeric? True


/tmp/ipykernel_36660/2268410240.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in employees.select_dtypes(include="object"):


**Findings:** 15 rows x 9 columns; every numeric-looking column except `EmployeeID` is stored as text; `Salary` has 2 missing values plus the text `"unknown"` (3 unusable salaries after conversion); `EmployeeID` 103 is repeated; department names are inconsistent (case, spaces, typos); one age is `150`; `HireDate` is text; `LastName`/`Department` had stray spaces; `PerformanceScore` had one string value, which can be converted without loss.

## Task 2 — Handle missing salaries

In [5]:
# errors="coerce" turns "unknown" (and anything unparsable) into NaN
employees["Salary"] = pd.to_numeric(employees["Salary"], errors="coerce")

missing = employees.isna().sum()
print(pd.DataFrame({"missing": missing, "percent": (missing / len(employees) * 100).round(1)}))

                  missing  percent
EmployeeID              0      0.0
FirstName               0      0.0
LastName                0      0.0
Department              0      0.0
Gender                  0      0.0
Age                     0      0.0
Salary                  3     20.0
HireDate                0      0.0
PerformanceScore        0      0.0


In [6]:
# dropna experiments (on copies, original untouched)
print("axis=0 (drop rows, default):", employees.dropna(axis=0).shape)
print("axis=1 (drop columns):      ", employees.dropna(axis=1).shape)
print("how='all':                  ", employees.dropna(how="all").shape)
print("subset=['Salary']:          ", employees.dropna(subset=["Salary"]).shape)
print("thresh=9 (>=9 non-null):    ", employees.dropna(thresh=9).shape)
print("thresh=8:                   ", employees.dropna(thresh=8).shape)

axis=0 (drop rows, default): (12, 9)
axis=1 (drop columns):       (15, 8)
how='all':                   (15, 9)
subset=['Salary']:           (12, 9)
thresh=9 (>=9 non-null):     (12, 9)
thresh=8:                    (15, 9)


`axis=0` drops rows with any NaN, `axis=1` drops whole columns that contain NaN (here it loses `Salary`). `how='any'` drops on one NaN, `how='all'` only if the whole row is empty (nothing changes). `subset` limits the check to chosen columns. `thresh=n` keeps rows with at least *n* non-null values. Dropping would lose employees, so the missing salaries are filled with the median instead (robust to outliers).

In [7]:
employees["Salary"] = employees["Salary"].fillna(employees["Salary"].median())
print(employees["Salary"].isna().sum(), "missing salaries left; median used:", employees["Salary"].median())

0 missing salaries left; median used: 71000.0


## Task 3 — Standardize department names

In [8]:
print("Before:", sorted(employees["Department"].unique()))
employees["Department"] = (
    employees["Department"].str.strip().str.replace(r"\s+", " ", regex=True).str.upper()
    .replace({"FINANCING": "FINANCE", "INFORMATION TECNOLOGY": "IT"})
)
employees["Department"] = employees["Department"].replace({"FINANCE": "Finance", "HR": "HR", "IT": "IT"})
print("After: ", sorted(employees["Department"].unique()))

Before: ['Finance', 'Financing', 'HR', 'IT', 'Information Tecnology', 'It', 'finance', 'it']
After:  ['Finance', 'HR', 'IT']


The incorrect values `Financing` and `Information Tecnology` were typos of existing departments, so they are mapped to `Finance` and `IT` (the names used in the `departments` table). Gender codes have the same case problem, so they are normalized too.

In [9]:
employees["Gender"] = employees["Gender"].str.strip().str.upper()
print(employees["Gender"].unique())

<StringArray>
['F', 'M']
Length: 2, dtype: str


## Task 4 — Remove duplicates

In [10]:
print("Exact duplicates:", employees.duplicated().sum())
print("Duplicate EmployeeID rows:\n", employees[employees["EmployeeID"].duplicated(keep=False)])

Exact duplicates: 1
Duplicate EmployeeID rows:
    EmployeeID FirstName  LastName Department Gender Age   Salary    HireDate  \
2         103   Gulnara  Mamatova         IT      F  29  71000.0  2024-03-01   
3         103   Gulnara  Mamatova         IT      F  29  71000.0  2024-03-01   

   PerformanceScore  
2               4.5  
3               4.5  


In [11]:
print("keep='first':", employees.drop_duplicates(keep="first").shape)
print("keep='last': ", employees.drop_duplicates(keep="last").shape)
print("keep=False:  ", employees.drop_duplicates(keep=False).shape, "(removes BOTH copies, so the employee disappears)")
print("subset=EmployeeID, keep='first':", employees.drop_duplicates(subset="EmployeeID", keep="first").shape)
print("subset=EmployeeID, keep='last': ", employees.drop_duplicates(subset="EmployeeID", keep="last").shape)
print("subset=EmployeeID, keep=False:  ", employees.drop_duplicates(subset="EmployeeID", keep=False).shape)

keep='first': (14, 9)
keep='last':  (14, 9)
keep=False:   (13, 9) (removes BOTH copies, so the employee disappears)
subset=EmployeeID, keep='first': (14, 9)
subset=EmployeeID, keep='last':  (14, 9)
subset=EmployeeID, keep=False:   (13, 9)


In [12]:
# keep="first": one record per employee
employees = employees.drop_duplicates(subset="EmployeeID", keep="first").reset_index(drop=True)
print(employees.shape)

(14, 9)


After normalizing text, rows 103 are exact duplicates (before cleaning `it` vs `IT` and `F` vs `f` hid this). `keep=False` would delete the employee entirely, so `keep='first'` is used.

## Task 5 — Convert data types

In [13]:
# 150 is an impossible age -> NaN, then nullable integer
employees["Age"] = pd.to_numeric(employees["Age"])
employees.loc[employees["Age"] > 100, "Age"] = np.nan
employees["Age"] = employees["Age"].astype("Int64")
print(employees["Age"].dtype); print(employees.dtypes)

Int64
EmployeeID            int64
FirstName               str
LastName                str
Department              str
Gender                  str
Age                   Int64
Salary              float64
HireDate                str
PerformanceScore    float64
dtype: object


In [14]:
employees[["Department", "Gender"]] = employees[["Department", "Gender"]].astype("category")
print(employees.dtypes)
print(employees["Department"].cat.categories.tolist())

EmployeeID             int64
FirstName                str
LastName                 str
Department          category
Gender              category
Age                    Int64
Salary               float64
HireDate                 str
PerformanceScore     float64
dtype: object
['Finance', 'HR', 'IT']


`object` is a generic container (here Python strings); each value is stored separately. `category` stores each distinct value once and keeps small integer codes per row, which saves memory and speeds up grouping for columns with few repeated values. `Int64` (capital I) is the nullable integer, so `Age` can keep a missing value without turning into float.

In [15]:
print(employees["PerformanceScore"].dtype)  # already float64 since Task 1
employees["PerformanceScore"] = pd.to_numeric(employees["PerformanceScore"], errors="coerce")

float64


## Task 6 — Parse hire dates

In [16]:
employees["HireDate"] = pd.to_datetime(employees["HireDate"], format="%Y-%m-%d")
employees["HireYear"] = employees["HireDate"].dt.year
employees["HireMonth"] = employees["HireDate"].dt.month
employees["HireWeekday"] = employees["HireDate"].dt.day_name()

today = pd.Timestamp.today().normalize()
employees["YearsWorked"] = ((today - employees["HireDate"]).dt.days / 365.25).round(1)
employees[["EmployeeID", "HireDate", "HireYear", "HireMonth", "HireWeekday", "YearsWorked"]].head()

,EmployeeID,HireDate,HireYear,HireMonth,HireWeekday,YearsWorked
0,101,2023-01-15,2023,1,Sunday,3.8
1,102,2022-06-10,2022,6,Friday,4.4
2,103,2024-03-01,2024,3,Friday,2.7
3,105,2021-11-20,2021,11,Saturday,4.9
4,106,2023-08-05,2023,8,Saturday,3.2


## Task 7 — Rename columns and merge datasets

In [17]:
employees = employees.rename(columns={"Salary": "MonthlySalary", "HireDate": "EmploymentStartDate"})
merged = employees.merge(departments, left_on="Department", right_on="DepartmentName", how="left")
print(employees.shape, "->", merged.shape)
print("Employees without department match:", merged["DepartmentID"].isna().sum())
merged[["EmployeeID", "FirstName", "Department", "DepartmentID", "Location"]].head()

(14, 13) -> (14, 16)
Employees without department match: 0


,EmployeeID,FirstName,Department,DepartmentID,Location
0,101,Aizura,IT,1,Bishkek
1,102,Bakyt,HR,2,Osh
2,103,Gulnara,IT,1,Bishkek
3,105,Nurlan,Finance,3,Bishkek
4,106,Aigul,HR,2,Osh


`how='left'` keeps every employee. The row count is unchanged (each department name is unique in `departments`), so the merge created no duplicates, and no employee lacks a department.

## Task 8 — Final check and summary

In [18]:
merged.info()
print("shape:", merged.shape)
print("missing:\n", merged.isna().sum()[lambda s: s > 0])
print("duplicate rows:", merged.duplicated().sum(), "| duplicate EmployeeID:", merged["EmployeeID"].duplicated().sum())
print("departments:", sorted(merged["Department"].unique()))

<class 'pandas.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   EmployeeID           14 non-null     int64         
 1   FirstName            14 non-null     str           
 2   LastName             14 non-null     str           
 3   Department           14 non-null     str           
 4   Gender               14 non-null     category      
 5   Age                  13 non-null     Int64         
 6   MonthlySalary        14 non-null     float64       
 7   EmploymentStartDate  14 non-null     datetime64[us]
 8   PerformanceScore     14 non-null     float64       
 9   HireYear             14 non-null     int32         
 10  HireMonth            14 non-null     int32         
 11  HireWeekday          14 non-null     str           
 12  YearsWorked          14 non-null     float64       
 13  DepartmentID         14 non-null     int64      

### Summary

**Problems found:** text-typed numbers and dates; missing and `"unknown"` salaries; a duplicated employee (ID 103); inconsistent department names (case, spaces, `Financing`, `Information Tecnology`) and gender codes; stray spaces; an impossible age (`150`, set to missing, so `Age` keeps 1 missing value); one string in `PerformanceScore`.

**Methods used:** `info`/`isna`/`notna` to inspect; `str.strip`/`str.upper`/`replace` for text; `pd.to_numeric(errors="coerce")` and `fillna(median)` for salary; `dropna(axis, how, subset, thresh)` to compare options; `drop_duplicates(subset, keep)`; `astype("Int64")` / `"category"`; `pd.to_datetime` and `.dt` accessors; `rename` and `merge(how="left")`.

**Why it matters:** wrong types break calculations (averages of text salaries), inconsistent labels split one group into many in a group-by, duplicates inflate headcounts, and impossible values distort statistics. Clean data makes the later analysis trustworthy.